# Décompression et fusion des dalles Lidar HD

Ce notebook vérifie que `processing.lidar.merge.merge_laz_tiles` décompresse et fusionne les dalles LAZ (COPC) téléchargées dans `data/raw/raster/lidar/` en un seul fichier LAS non compressé, sans perte de points ni de dimensions (classification IGN comprise).

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

from processing.lidar.merge import merge_laz_tiles

## 1. Fusion des dalles LAZ en un seul fichier LAS

In [ ]:
lidar_dir = repo_root / "data" / "raw" / "raster" / "lidar"
output_path = lidar_dir / "nuage_points.las"

tiles = sorted(lidar_dir.glob("*.laz"))
print(f"{len(tiles)} dalle(s) source :")
for tile in tiles:
    print(" -", tile.name, f"({tile.stat().st_size / 1_000_000:.1f} Mo)")

result = merge_laz_tiles(lidar_dir, output_path)
print("\nFichier fusionne :", result)
print(f"Taille : {result.stat().st_size / 1_000_000:.1f} Mo (non compresse)")

## 2. Vérification : nombre de points, dimensions, classification

In [ ]:
import json

import pdal

pipeline = pdal.Pipeline(json.dumps([{"type": "readers.las", "filename": str(result)}]))
count = pipeline.execute()
arr = pipeline.arrays[0]

print("Points fusionnes :", count)
print("Dimensions disponibles :", arr.dtype.names)
print("Emprise X :", arr["X"].min(), "-", arr["X"].max())
print("Emprise Y :", arr["Y"].min(), "-", arr["Y"].max())
print("Emprise Z :", arr["Z"].min(), "-", arr["Z"].max())

import numpy as np

classes, counts = np.unique(arr["Classification"], return_counts=True)
print("\nRepartition par classe LAS (classification IGN deja fournie dans le LAZ source) :")
for cls, n in zip(classes, counts):
    print(f"  classe {cls} : {n} points")

## 3. Aperçu du nuage de points (coupe, coloré par classification)

Un sous-échantillon est utilisé pour l'affichage (le nuage complet compte plusieurs dizaines de millions de points).

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
sample_idx = rng.choice(len(arr), size=min(200_000, len(arr)), replace=False)
sample = arr[sample_idx]

fig, ax = plt.subplots(figsize=(8, 8))
scatter = ax.scatter(sample["X"], sample["Y"], c=sample["Classification"], s=0.5, cmap="tab10")
ax.set_aspect("equal")
ax.set_title("Nuage de points fusionne (vue en plan, colore par classe LAS)")
legend = ax.legend(*scatter.legend_elements(), title="Classe", loc="upper left", bbox_to_anchor=(1, 1))
ax.add_artist(legend)
plt.show()